# 0. Imports

## 0.1 Packages

The packages used were the ones that we found to fit better for the tasks done:

>*asyncio* and *nest_asyncio*: used for the taxonomy updates loops to be done asynchronously and concurrently, saving up time on running the code.

>*tenacity*: used to guarantee that the extraction code was not stopped if an error occurred, and if it did occurred to run again for a predefined amount of times with a waiting time between them.

>*aiohttp*: used to call the different API's, in this case the GBIF and Global Names Verifier APIs, while working with *asyncio* and *nest_asyncio*.

>*pandas*: used to format the data on to the final schema.

In [1]:
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import pandas as pd

## 0.2 Data

All data was previously manually retrieved and fitted to a predefined schema.

For the Observation data, here on after referred as *ObsList*, it corresponded to:

>*Species*: Species name used on source, corrected for typos and manually confirmed on Global Names Verifier.

>*NAME_0*: The corresponding name of the area that was observed at the predefined regions dataset.

>*Realm*: The biogeographic realm of the observation fitting the updates defined at 10.1126/science.1228282.

>*Cryptogenic*: A binary variable that corresponded to the cases where the species origin was described as unknown or nativeness uncertain for the observed region. 

>*Dispersal*: A binary variable that stated if the species arrived by dispersal either from an introduced region or by natural dispersal from their native range.

>*Eradicated*: A binary variable that stated if the species was registered as eradicated to a previous establishment at that region.

>*IntentionalRelease*: A binary variable that stated if the species was intentionally released, either for biological control or for any other reason. 

>*Introduced*: A binary variable that recorded if the species was considered to have arrived by human action. For the cases that the species dispersed (*Dispersal* = 1) from a region that it was not native to it kept the introduced status (Introduced = 1).

>*Established*: A binary variable stating if the species was considered to have a stable population at a certain region independently if it arrived there by dispersal or by man mediated introductions.

>*ReportedFirstYear*: The year of the oldest record according to the source reference.

>*Reference*: The source reference.

>*ReferenceYear*: The year of the source reference, keeping in consideration that a species that was established at a certain point of time can now be eradicated.

In [2]:
ObsList = pd.read_csv(r"../Data Raw/ObsList.csv", sep=";")
NativeData = pd.read_csv(r"../Data Raw/NativeData.csv", sep=";")

# 1. Taxonomy Standardization

The standardization of the taxonomy was done in 3 main steps: 

1. GBIF Standardization: the raw names on ObsList were ran onto the GBIF API in a first instance to standardize according to what were considering as accepted names, keeping the names that were not considered as accepted.

2. Global Names Verifier Standardization: to be assure that the most names were captured and that they could be standardized, removing possible false duplicates that were considered as accepted by GBIF as separate species while in fact being one species only, we repeated the process with Global Names Verifier API. 

3. GBIF Filtration: After the 2 standardization steps we filtered keeping the accepted name as the one that was found for the standardization on GBIF.

## 1.1. Code

Each extractor was composed by 3 parts: Species List Retainer; Session Creator; Extractor.

- Notation (API_Function_n)
    - API : the API used;
    - Function: Species (Species List Retainer), Sessions (Session Creator), Extractor (Extractor);
    - n: 1 for standardization; 2 for filtration

- Steps:
    - >The Species List Retainer: It takes 2 parameters to start, the species and a session, using the defined session it will open the url for the API and try to retrieved the canonical accepted name for the given species. If there is none accepted species it will return the species name if it has the standardization role or nothing if it is a filter.

    - >The Session Creator: It takes a list of species and creates a session for each. Using the Species List Retainer it creates a list of tasks to be done asynchronously to be called in the future.

    - >The Extractor: It calls the Session Creator, running the list of tasks returning the results for each species in the list using the function on the Species List Retainer.

### 1.1.1. GBIF: Names Check

#### 1.1.1.1. Standardization

In [3]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_1(session, species): 

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return species
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_1(species_list))

#### 1.1.1.2. Filter

In [4]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_2(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return None
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_2(species_list))

### 1.1.2. Global Names Verifier: Cross-check

#### 1.1.2.1. Standardization

In [5]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_1(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name if accepted_name != "" else species
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_1(species_list))

## 1.2. Data

### 1.2.1. Observation Data

Before doing the standardization we kept only the first two words on the *Species* field to keep only the "Genus + Specific Epithet", dropping any possible subspecies that had been recorded.

We also cleaned for *¬†* characters that resulted from different file types formatting.

In [6]:
ObsList['Species'] = ObsList['Species'].apply(lambda x: ' '.join(x.split()[:2]))
ObsList['Species'] = ObsList['Species'].str.replace('¬†', ' ', regex=False)

In [7]:
ObsList['AcceptedSpecies'] = GBIF_Extractor_2(VNF_Extractor_1(GBIF_Extractor_1(ObsList['Species'])))
ObsList['AcceptedSpecies'] = ObsList['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

In [8]:
ObsList[ObsList['AcceptedSpecies'].isna()]['Species'].unique()

array(['Heliocheilus cystiphora', 'Rheumaptera affirmata',
       'Spodoptera sunia', 'Heliocontia margana', 'Trissodoris guamensis',
       'Prospalta dolorosa', 'Phalaenophana fadusalis',
       'Opodiphthera eucalypti', 'Euphaedra temeraria', 'Orgyia basalis'],
      dtype=object)

In [9]:
ObsList[ObsList['AcceptedSpecies'].isna()].shape[0]


13

Considering that there were only 10 taxonomy cases, corresponding to only 13 registers, that could not be resolved by the applied methodology it was opted to drop these cases. 

In [10]:
ObservationsClean = ObsList.copy()
ObservationsClean.dropna(subset=['AcceptedSpecies'], inplace=True)
ObservationsClean[ObservationsClean['AcceptedSpecies'].isna()]

,Species,NAME_0,Realm,Cryptogenic,Dispersal,Eradicated,IntentionalRelease,Introduced,Established,ReportedFirstYear,Reference,ReferenceYear,AcceptedSpecies


In [11]:
ObservationsClean.reset_index(drop=True, inplace=True)
ObservationsClean.to_csv(r'../Transformed Data/RecordsClean.csv', sep =';', encoding='utf-8', index=False)

### 1.2.2. Native Distribution Data

We repeated the standardization methodology for the native distribution data that was previously recorded for each species that was introduced and established.

In [12]:
NativeData['Species'] = NativeData['Species'].apply(lambda x: ' '.join(x.split()[:2]))

In [13]:
NativeData['AcceptedSpecies'] = GBIF_Extractor_2(VNF_Extractor_1(GBIF_Extractor_1(NativeData['Species'])))
NativeData['AcceptedSpecies'] = NativeData['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

In [14]:
NativeData[NativeData['AcceptedSpecies'].isna()]['Species'].unique()

array([], dtype=object)

In [15]:
NativeDataClean = NativeData[NativeData['AcceptedSpecies'].isin(ObservationsClean['AcceptedSpecies'])].copy()

In [16]:
NativeDataClean.reset_index(drop=True, inplace=True)
NativeDataClean.to_csv(r'../Transformed Data/NativeDataClean.csv', sep =';', encoding='utf-8', index=False)